In [1]:
import time
import pickle
import numpy as np
import pandas as pd

from scipy.stats import uniform, randint
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV
)

In [2]:
seed = 42

### **Data Loading**

In [3]:
file_path = "data/train.csv"

train = pd.read_csv(file_path)
train.head()

,Team_Size,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Stakeholder_Count,Past_Similar_Projects,External_Dependencies_Count,Change_Request_Frequency,Team_Turnover_Rate,Vendor_Reliability_Score,...,Risk_Management_Maturity_None,Team_Colocation_Fully Colocated,Team_Colocation_Fully Remote,Team_Colocation_Hybrid,Team_Colocation_Partially Colocated,Documentation_Quality_Basic,Documentation_Quality_Excellent,Documentation_Quality_Good,Documentation_Quality_Poor,Risk_Level
0,0.919707,1.566060,1.698262,1.710485,1.301350,-1.140618,2.395344,0.079515,2.471318,0.474452,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,3
1,0.702913,1.185257,0.978376,1.616106,0.403204,-1.140618,-0.080692,0.702951,1.386302,-0.076372,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,3
2,-0.272656,-0.330071,-0.029464,0.267840,0.178667,0.555168,-0.080692,-0.156379,-0.904288,-0.137575,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1
3,-0.597846,-0.840667,-0.029464,0.061105,-1.393089,0.555168,-0.699701,-0.990435,-0.783730,0.046033,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0
4,2.545656,2.528718,1.266330,-0.370340,2.424033,-0.010094,-1.318710,-1.260029,-1.265960,1.086479,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0


In [4]:
X_train = train.drop(columns=["Risk_Level"])
y_train = train["Risk_Level"].copy()

### **Model Training**

In [5]:
# Logistic Regression

t0 = time.time()

logit = GridSearchCV(
    LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=seed
    ),
    param_grid={
        "C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]
    },
    scoring="recall_macro",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=seed),
    n_jobs=-1,
    verbose=1
)
logit.fit(X_train, y_train)

tf = time.time()

print(f"Fit Time: {int((tf - t0) // 60)}m {(tf - t0) % 60:.1f}s\n")
print(f"Best Parameters: {logit.best_params_} - Logistic Regression")

with open(f"models/LogisticRegression.pkl", "wb") as f:
    pickle.dump(logit, f)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Fit Time: 0m 4.0s

Best Parameters: {'C': 10000.0} - Logistic Regression


In [6]:
# Linear Discriminant Analysis

t0 = time.time()

lda = GridSearchCV(
    LinearDiscriminantAnalysis(
        solver="lsqr"
    ),
    param_grid={
        "shrinkage": [None, "auto", 0.1, 0.3, 0.5, 0.7, 0.9]
    },
    scoring="recall_macro",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=seed),
    n_jobs=-1,
    verbose=1
)
lda.fit(X_train, y_train)

tf = time.time()

print(f"Fit Time: {int((tf - t0) // 60)}m {(tf - t0) % 60:.1f}s\n")
print(f"Best Parameters: {lda.best_params_} - Linear Discriminant Analysis")

with open(f"models/LDA.pkl", "wb") as f:
    pickle.dump(lda, f)

Fitting 5 folds for each of 7 candidates, totalling 35 fits
Fit Time: 0m 0.4s

Best Parameters: {'shrinkage': 0.1} - Linear Discriminant Analysis


In [7]:
# Support Vector Machine (RBF)

t0 = time.time()

svm_rbf = GridSearchCV(
    SVC(
        kernel="rbf",
        class_weight="balanced",
    ),
    param_grid={
        "C": [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
        "gamma": ["scale", "auto", 0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]
    },
    scoring="recall_macro",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=seed),
    n_jobs=-1,
    verbose=1
)
svm_rbf.fit(X_train, y_train)

tf = time.time()

print(f"Fit Time: {int((tf - t0) // 60)}m {(tf - t0) % 60:.1f}s\n")
print(f"Best Parameters: {svm_rbf.best_params_} - Support Vector Machines (RBF)")

with open(f"models/SVM-RBF.pkl", "wb") as f:
    pickle.dump(svm_rbf, f)

Fitting 5 folds for each of 64 candidates, totalling 320 fits
Fit Time: 0m 47.2s

Best Parameters: {'C': 100.0, 'gamma': 0.001} - Support Vector Machines (RBF)


In [8]:
# Support Vector Machine (Linear)

t0 = time.time()

svm_linear = GridSearchCV(
    SVC(
        kernel="linear",
        class_weight="balanced",
    ),
    param_grid={
        "C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
    },
    scoring="recall_macro",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=seed),
    n_jobs=-1,
    verbose=1
)
svm_linear.fit(X_train, y_train)

tf = time.time()

print(f"Fit Time: {int((tf - t0) // 60)}m {(tf - t0) % 60:.1f}s\n")
print(f"Best Parameters: {svm_linear.best_params_} - Support Vector Machines (Linear)")

with open(f"models/SVM-Linear.pkl", "wb") as f:
    pickle.dump(svm_linear, f)

Fitting 5 folds for each of 7 candidates, totalling 35 fits
Fit Time: 7m 57.7s

Best Parameters: {'C': 10.0} - Support Vector Machines (Linear)


In [9]:
# Random Forest

t0 = time.time()

rf = RandomizedSearchCV(
    RandomForestClassifier(    
        criterion="gini",
        bootstrap=True,
        class_weight="balanced",
        random_state=seed
    ),
    param_distributions={
        "n_estimators": randint(200, 1000),
        "max_depth": [None] + list(randint.rvs(10, 60, size=10)),
        "min_samples_split": randint(2, 20),
        "min_samples_leaf": randint(1, 10),
        "max_features": ["sqrt", "log2", None],
    },
    n_iter=100,
    scoring="recall_macro",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=seed),
    n_jobs=-1,
    verbose=1,
    random_state=seed
)
rf.fit(X_train, y_train)

tf = time.time()

print(f"Fit Time: {int((tf - t0) // 60)}m {(tf - t0) % 60:.1f}s\n")
print(f"Best Parameters: {rf.best_params_} - Random Forest")

with open(f"models/RandomForest.pkl", "wb") as f:
    pickle.dump(rf, f)

Fitting 5 folds for each of 100 candidates, totalling 500 fits


Fit Time: 9m 37.5s

Best Parameters: {'max_depth': 44, 'max_features': 'log2', 'min_samples_leaf': 7, 'min_samples_split': 10, 'n_estimators': 840} - Random Forest


In [10]:
# Multi-Layer Perceptron

t0 = time.time()

mlp = RandomizedSearchCV(
    MLPClassifier(
        max_iter=10000,
        random_state=seed, 
    ),
    param_distributions={
    "hidden_layer_sizes": [
        (randint.rvs(50, 200),),
        (randint.rvs(80, 200), randint.rvs(30, 150)),
    ],
    "activation": ["relu", "tanh"],
    "solver": ["adam", "sgd", "lbfgs"],
    "alpha": uniform(1e-5, 1e-2),
    "learning_rate": ["constant", "adaptive"],
    "learning_rate_init": uniform(1e-4, 5e-3),
    "batch_size": [32, 64, 128, 256],
    },
    n_iter=100,
    scoring="recall_macro",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=seed),
    n_jobs=-1,
    verbose=1,
    random_state=seed
)
mlp.fit(X_train, y_train)

tf = time.time()

print(f"Fit Time: {int((tf - t0) // 60)}m {(tf - t0) % 60:.1f}s\n")
print(f"Best Parameters: {mlp.best_params_} - Multi-Layer Perceptron")

with open(f"models/MLP.pkl", "wb") as f:
    pickle.dump(mlp, f)


Fitting 5 folds for each of 100 candidates, totalling 500 fits
Fit Time: 9m 15.6s

Best Parameters: {'activation': 'tanh', 'alpha': 0.007861759613930135, 'batch_size': 128, 'hidden_layer_sizes': (172, 73), 'learning_rate': 'constant', 'learning_rate_init': 0.005016154429033941, 'solver': 'adam'} - Multi-Layer Perceptron
